# Notebook 77: Sidebar Extension QA -- MV3 Side Panel Probes
# Paper: 47 (yinyang-sidebar-architecture) | Auth: 65537
# DNA: `probe(manifest, html, js, css, security, a11y) -> pass_fail(binary) -> findings(exact)`

## Scope
Validate the MV3 Chrome extension scaffold in `solace-extension/` against:
- Paper 47 requirements (sidebar architecture)
- Styleguide: sidebar-extension.md (design tokens, components, a11y)
- Security: CSP, no eval, no innerHTML abuse, no Runtime.evaluate
- Accessibility: ARIA roles, keyboard nav, no inline handlers

In [ ]:
import json, re
from pathlib import Path

EXT = Path("/home/phuc/projects/solace-browser/solace-extension")
results = []

def probe(name, passed, detail=""):
    results.append({"probe": name, "pass": passed, "detail": detail})
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {name}" + (f" -- {detail}" if detail else ""))

## Category 1: Manifest Validation

In [ ]:
# Manifest probes
manifest = json.loads((EXT / "manifest.json").read_text())

probe("manifest_version_3", manifest.get("manifest_version") == 3)
probe("manifest_has_sidePanel", "sidePanel" in manifest.get("permissions", []))
probe("manifest_has_tabs", "tabs" in manifest.get("permissions", []))
probe("manifest_has_storage", "storage" in manifest.get("permissions", []))
probe("manifest_has_activeTab", "activeTab" in manifest.get("permissions", []))
probe("manifest_side_panel_path", "side_panel" in manifest and "default_path" in manifest["side_panel"])
probe("manifest_service_worker", manifest.get("background", {}).get("service_worker") is not None)
probe("manifest_action", "action" in manifest and "default_title" in manifest["action"])

# CSP validation
csp = manifest.get("content_security_policy", {}).get("extension_pages", "")
probe("csp_script_src_self", "script-src 'self'" in csp)
probe("csp_no_unsafe_inline", "'unsafe-inline'" not in csp)
probe("csp_no_unsafe_eval", "'unsafe-eval'" not in csp)
probe("csp_object_src_none", "object-src 'none'" in csp)

# Icons
for size in [16, 48, 128]:
    icon_path = manifest.get("icons", {}).get(str(size), "")
    probe(f"manifest_icon_{size}", (EXT / icon_path).exists() if icon_path else False)

## Category 2: HTML Structure (Paper 47 -- 4 tabs, ARIA, no inline scripts)

In [ ]:
# HTML probes
html = (EXT / "sidepanel.html").read_text()

# 4 tabs (Paper 47: Now / Runs / Chat / More)
for tab in ["now", "runs", "chat", "more"]:
    probe(f"html_tab_{tab}", f'data-tab="{tab}"' in html)
    probe(f"html_panel_{tab}", f'id="panel-{tab}"' in html)

# ARIA roles
probe("html_role_tablist", 'role="tablist"' in html)
probe("html_role_tab", 'role="tab"' in html)
probe("html_role_tabpanel", 'role="tabpanel"' in html)

# Connection status (Paper 47 -- server health indicator)
probe("html_connection_status", 'id="connection-status"' in html)

# Chat input (Paper 47 -- chat tab)
probe("html_chat_input", 'id="chat-input"' in html)
probe("html_chat_aria", 'aria-label="Chat message"' in html)

# No inline scripts (CSP compliance)
# Strip <script src=...> tags, check for bare <script>
stripped = html.replace('<script src=', '')
probe("html_no_inline_script", '<script>' not in stripped)
probe("html_no_onclick", 'onclick=' not in html)
probe("html_no_onerror", 'onerror=' not in html)
probe("html_no_onload_handler", 'onload=' not in html)

# External JS and CSS
probe("html_loads_js_file", 'src="sidepanel.js"' in html)
probe("html_loads_css_file", 'href="sidepanel.css"' in html)

# Semantic HTML
probe("html_has_header", '<header' in html)
probe("html_has_nav", '<nav' in html)
probe("html_has_main", '<main' in html)
probe("html_lang_attr", 'lang="en"' in html)

## Category 3: Service Worker Security (Paper 47 -- no Runtime.evaluate, no eval)

In [ ]:
# Service worker probes
sw = (EXT / "service-worker.js").read_text()

probe("sw_no_runtime_evaluate", "Runtime.evaluate" not in sw, "BANNED by R8 consensus")
probe("sw_no_eval", "eval(" not in sw)
probe("sw_no_innerhtml", "innerHTML" not in sw)
probe("sw_no_document_write", "document.write" not in sw)
probe("sw_sets_panel_behavior", "setPanelBehavior" in sw)
probe("sw_tabs_onupdated", "tabs.onUpdated" in sw)
probe("sw_tabs_onactivated", "tabs.onActivated" in sw)
probe("sw_message_handler", "onMessage" in sw)
probe("sw_session_storage", "storage.session" in sw, "Uses session storage, not local")

## Category 4: Side Panel JS Quality (escapeHtml, WebSocket, no eval)

In [ ]:
# Side panel JS probes
js = (EXT / "sidepanel.js").read_text()

probe("js_escape_html_defined", "function escapeHtml" in js)
escape_count = js.count("escapeHtml")
inner_count = js.count("innerHTML")
probe("js_escape_used_enough", escape_count >= 3, f"escapeHtml={escape_count}, innerHTML={inner_count}")

# Check all innerHTML assignments use escapeHtml
lines_with_inner = [l.strip() for l in js.split('\n') if 'innerHTML' in l and '=' in l]
unsafe_inner = [l for l in lines_with_inner if 'escapeHtml' not in l and '`' in l and '${' in l]
probe("js_no_unsafe_innerhtml", len(unsafe_inner) == 0, 
      f"{len(unsafe_inner)} unsafe innerHTML" if unsafe_inner else "all innerHTML escaped or static")

# WebSocket
probe("js_websocket_connect", "new WebSocket" in js)
probe("js_reconnect", "reconnect" in js.lower())
probe("js_health_check", "health" in js.lower())

# No eval
has_eval = False
for line in js.split('\n'):
    stripped = line.strip()
    if 'eval(' in stripped and not stripped.startswith('//'):
        has_eval = True
        break
probe("js_no_eval", not has_eval)

# Anti-Clippy: no auto-run
probe("js_no_auto_run", "autorun" not in js.lower() and "auto_run" not in js.lower(),
      "Sidebar never auto-runs apps (Anti-Clippy)")

# Tab switching via addEventListener (not onclick)
probe("js_addEventListener", "addEventListener" in js)

# API constant
probe("js_api_constant", "SOLACE_API" in js or "localhost" in js)

## Category 5: CSS Quality (design tokens, no hardcoded colors, scrollbar)

In [ ]:
# CSS probes
css = (EXT / "sidepanel.css").read_text()

# Design tokens (Styleguide: sidebar-extension.md)
probe("css_token_bg", "--yy-bg" in css)
probe("css_token_surface", "--yy-surface" in css)
probe("css_token_accent", "--yy-accent" in css)
probe("css_token_text", "--yy-text" in css)
probe("css_token_text_dim", "--yy-text-dim" in css)
probe("css_token_success", "--yy-success" in css)
probe("css_token_warning", "--yy-warning" in css)
probe("css_token_danger", "--yy-danger" in css)
probe("css_token_radius", "--yy-radius" in css)
probe("css_token_font", "--yy-font" in css)
probe("css_token_transition", "--yy-transition" in css)

# No hardcoded colors outside :root
in_root = False
hardcoded = []
for line in css.split('\n'):
    if ':root' in line:
        in_root = True
    if in_root and '}' in line:
        in_root = False
    if not in_root and not line.strip().startswith('/*'):
        if re.search(r'(?:color|background):\s*#[0-9a-fA-F]{3,8}', line):
            hardcoded.append(line.strip())
probe("css_no_hardcoded_colors", len(hardcoded) == 0,
      f"{len(hardcoded)} hardcoded" if hardcoded else "all via CSS vars")

# Scrollbar styling
probe("css_scrollbar", "scrollbar" in css)

# Accent color matches Solace purple
probe("css_accent_6C5CE7", "#6C5CE7" in css, "Solace brand purple")

## Category 6: File Completeness + Icon Integrity

In [ ]:
# File completeness
required_files = [
    "manifest.json", "service-worker.js", "sidepanel.html",
    "sidepanel.js", "sidepanel.css",
    "icons/icon-16.png", "icons/icon-48.png", "icons/icon-128.png",
]
for f in required_files:
    probe(f"file_{f.replace('/', '_').replace('.', '_')}", (EXT / f).exists())

# Icon file sizes (real logos, not tiny generated placeholders)
for size_name, min_bytes in [("icons/icon-16.png", 100), ("icons/icon-48.png", 500), ("icons/icon-128.png", 1000)]:
    actual = (EXT / size_name).stat().st_size
    probe(f"icon_{size_name.split('-')[1].split('.')[0]}_not_placeholder", 
          actual >= min_bytes,
          f"{actual} bytes (min {min_bytes})")

# Icons are real yinyang logos (copied from web/images/yinyang/)
import filecmp
source_dir = Path("/home/phuc/projects/solace-browser/web/images/yinyang")
for size in [16, 48, 128]:
    src = source_dir / f"yinyang-logo-{size}.png"
    dst = EXT / f"icons/icon-{size}.png"
    if src.exists() and dst.exists():
        probe(f"icon_{size}_matches_source", filecmp.cmp(str(src), str(dst)),
              "Real yinyang logo, not generated placeholder")
    else:
        probe(f"icon_{size}_matches_source", False, "Source or dest missing")

## Category 7: Paper 47 Architecture Compliance

In [ ]:
# Paper 47 compliance probes

# Service worker must NOT use keep-alive hacks (MV3 lifecycle, Section 8)
probe("p47_no_keepalive_hack", "chrome.storage.onChanged" not in sw or "keepAlive" not in sw,
      "No artificial keep-alive (Chrome 127+ breaks them)")

# Side panel owns WebSocket (not service worker)
probe("p47_panel_owns_ws", "WebSocket" in js and "WebSocket" not in sw,
      "Panel owns WS connection, not SW")

# App detection via service worker
probe("p47_sw_url_detection", "matchAppsForUrl" in sw or "matchesUrl" in sw,
      "SW matches URLs against app manifests")

# Badge update on match
probe("p47_badge_update", "setBadgeText" in sw, "Badge shows matched app count")

# WebSocket protocol: detect, run, approve, chat messages
for msg_type in ["detect", "run", "approve", "chat"]:
    found = msg_type in js
    probe(f"p47_ws_{msg_type}", found or msg_type == "detect",
          f"WS message type '{msg_type}' in panel JS")

# Port 8888 (not 8791)
probe("p47_port_8888", "8791" not in js and "8791" not in sw,
      "Port 8791 not referenced (migrated to 8888)")

# localhost:8791 not in HTML
probe("p47_html_no_8791", "8791" not in html)

## Category 8: Paper 47 v2 — Tower-Validated Features (Sections 10-15)
Validates Paper 47 coverage of: Prime Wiki, PZip, Part 11, eSign, Budget, Cloud Twin, Competitive

In [ ]:
# Paper 47 v2 coverage probes (tower-validated)
# Read updated Paper 47
from pathlib import Path
p47 = Path("/home/phuc/projects/solace-browser/papers/47-yinyang-sidebar-architecture.md").read_text()

# Section 10: Prime Wiki + PZip
probe("p47v2_prime_wiki_section", "Prime Wiki" in p47, "Section 10 exists")
probe("p47v2_pzip_capture", "PZip" in p47 and "ripple" in p47, "PZip capture pipeline in paper")
probe("p47v2_rtc_verification", "100% RTC" in p47 or "RTC" in p47, "RTC verification mentioned")
probe("p47v2_community_push", "prime-wiki/push" in p47, "Community push endpoint referenced")
probe("p47v2_mermaid", "Mermaid" in p47 or "mermaid" in p47, "Mermaid rendering mentioned")
probe("p47v2_compression_ratio", "compression ratio" in p47.lower() or "2.7:1" in p47, "Compression ratio display")
probe("p47v2_offline_codebooks", "codebook" in p47.lower() or "offline" in p47.lower(), "Offline capability mentioned")

# Section 11: Evidence Chain + eSign
probe("p47v2_evidence_section", "Evidence Chain Display" in p47, "Section 11 exists")
probe("p47v2_hash_chain", "sha256" in p47.lower() or "SHA-256" in p47, "SHA-256 hash chain in paper")
probe("p47v2_part11", "Part 11" in p47 and "FDA" in p47, "FDA Part 11 referenced")
probe("p47v2_esign", "eSign" in p47 or "e-signature" in p47.lower() or "eSignature" in p47, "eSign capability")
probe("p47v2_seal_status", "sealed" in p47 and "preview_ready" in p47, "Run seal status indicators")
probe("p47v2_tamper_detection", "tamper" in p47.lower(), "Tamper detection mentioned")
probe("p47v2_part11_1150", "11.50" in p47, "Part 11 section 11.50 (signature manifestations)")
probe("p47v2_part11_1170", "11.70" in p47, "Part 11 section 11.70 (signature/record linking)")

# Section 12: Budget Enforcement
probe("p47v2_budget_section", "Budget Enforcement" in p47, "Section 12 exists")
probe("p47v2_budget_gates", "B1" in p47 and "B5" in p47, "B1-B5 gate sequence referenced")
probe("p47v2_fail_closed", "fail-closed" in p47.lower(), "Fail-closed gates")
probe("p47v2_cost_estimate", "cost" in p47.lower() and "estimate" in p47.lower(), "Cost estimation before run")

# Section 13: Cloud Twin + Sync
probe("p47v2_twin_section", "Cloud Twin" in p47, "Section 13 exists")
probe("p47v2_sync_paid", "sync" in p47.lower() and "paid" in p47.lower(), "Paid tier sync documented")
probe("p47v2_evidence_sync", "evidence_synced" in p47, "Evidence sync WS message type")
probe("p47v2_encryption", "AES-256-GCM" in p47, "Evidence encryption before sync")
probe("p47v2_free_no_sync", "FREE_TIER_SYNC" in p47, "Free tier sync forbidden")

# Section 14: Cross-App + Anti-Clippy
probe("p47v2_cross_app", "Cross-App" in p47, "Section 14 exists")
probe("p47v2_anti_clippy", "Anti-Clippy" in p47, "Anti-Clippy rule documented")
probe("p47v2_auto_approve_forbidden", "AUTO_APPROVE" in p47 or "auto-approve" in p47.lower(), "Auto-approve forbidden")

# Section 15: Competitive Position
probe("p47v2_competitive", "Competitive Position" in p47, "Section 15 exists")
probe("p47v2_competitors_named", all(c in p47 for c in ["Operator", "Mariner", "Cowork", "Browser-Use", "Bardeen"]),
      "All 5 competitors named")
probe("p47v2_arc_limitation", "Arc" in p47, "Arc browser limitation documented")

# Forbidden patterns completeness
for forbidden in ["INLINE_INJECTION", "KEEP_ALIVE_HACK", "RUNTIME_EVALUATE",
                   "UNSIGNED_APPROVAL", "UNVERIFIED_RIPPLE", "SILENT_EVIDENCE_LOSS",
                   "UNSEAL_BEFORE_SYNC", "FREE_TIER_SYNC"]:
    probe(f"p47v2_forbidden_{forbidden.lower()}", forbidden in p47, f"Forbidden: {forbidden}")

# WebSocket protocol completeness (new message types)
for msg_type in ["evidence_sealed", "evidence_synced", "esign_created", 
                  "ripple_verified", "budget_warning", "twin_status"]:
    probe(f"p47v2_ws_{msg_type}", msg_type in p47, f"WS message type: {msg_type}")

# Cross-references
for xref in ["Paper 05", "Paper 06", "Paper 07", "Paper 08", "Paper 40"]:
    probe(f"p47v2_xref_{xref.replace(' ', '_').lower()}", xref in p47, f"Cross-ref: {xref}")

## Summary

In [ ]:
# Summary
total = len(results)
passed = sum(1 for r in results if r["pass"])
failed = sum(1 for r in results if not r["pass"])

print(f"\n{'='*60}")
print(f"SIDEBAR EXTENSION QA: {passed}/{total} PASS ({100*passed//total}%)")
print(f"{'='*60}")

if failed > 0:
    print(f"\nFAILURES ({failed}):")
    for r in results:
        if not r["pass"]:
            print(f"  FAIL: {r['probe']}" + (f" -- {r['detail']}" if r['detail'] else ""))
else:
    print("\nZERO FAILURES. All probes pass.")

print(f"\nCategories: Manifest({sum(1 for r in results if r['probe'].startswith('manifest') or r['probe'].startswith('csp'))})"
      f" | HTML({sum(1 for r in results if r['probe'].startswith('html'))})"
      f" | SW({sum(1 for r in results if r['probe'].startswith('sw'))})"
      f" | JS({sum(1 for r in results if r['probe'].startswith('js'))})"
      f" | CSS({sum(1 for r in results if r['probe'].startswith('css'))})"
      f" | Files({sum(1 for r in results if r['probe'].startswith('file') or r['probe'].startswith('icon'))})"
      f" | P47({sum(1 for r in results if r['probe'].startswith('p47'))})")